# C题脚本逐段拆解 Notebook

这份 notebook 是根据 `c_problem_starter.py` 自动生成的学习版。

当前脚本采用 `pandas` + `DataFrame` 的组织方式，最值得你重点观察的是：

1. Excel 直接通过 `pandas.read_excel()` 读取；
2. 后续切片主要靠“列名”而不是列号；
3. 问题 1 / 2 / 3 的核心建模逻辑围绕同一张 `DataFrame` 展开。

推荐学习顺序：

- 先跑通 `load_dataframe()`；
- 再看问题 1 的指标筛选；
- 再看问题 2 的双层风险模型；
- 最后看问题 3 的干预优化。


## 这份 notebook 怎么看

这份 notebook 会按脚本原始顺序展开：

- 导入依赖
- 常量和列名映射
- 数据结构
- 数据读取
- 问题 1
- 问题 2
- 问题 3
- 命令行入口

我在关键函数后额外补了三类内容：

- `函数签名`
- `学习提示`
- `试运行`

这样你可以一边对照源码，一边看实际输出。


## 1. 导入依赖

这部分是当前脚本的基础环境。

这里会直接导入 `pandas`，说明它已经是这份脚本的默认依赖。


In [ ]:
from __future__ import annotations

import argparse
import json
import shutil
from dataclasses import dataclass
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text


### 学习提示

这里可以顺手记住：`pandas` 已经是这份脚本的默认依赖。

所以你后面阅读和运行时，都可以把 `DataFrame` 当作主线来理解。


## 2. 常量、列名映射和字段分组

这一段是当前脚本的“坐标系”。

最重要的几组信息是：

- `RAW_TO_ENGLISH`：把 Excel 中文列名映射成英文列名；
- `CONSTITUTION_SCORE_COLS`：九种体质积分列；
- `ACTIVITY_LAB_COLS`：活动能力和检验指标；
- `EARLY_WARNING_COLS` / `ALL_FEATURE_COLS`：问题 2 不同模型使用的列组。


In [ ]:
CONSTITUTION_NAMES = [
    "Balanced",
    "Qi_deficiency",
    "Yang_deficiency",
    "Yin_deficiency",
    "Phlegm_dampness",
    "Damp_heat",
    "Blood_stasis",
    "Qi_stagnation",
    "Special_diathesis",
]

RAW_TO_ENGLISH = {
    "样本ID": "sample_id",
    "体质标签": "constitution_tag",
    "平和质": "balanced_score",
    "气虚质": "qi_deficiency_score",
    "阳虚质": "yang_deficiency_score",
    "阴虚质": "yin_deficiency_score",
    "痰湿质": "phlegm_score",
    "湿热质": "damp_heat_score",
    "血瘀质": "blood_stasis_score",
    "气郁质": "qi_stagnation_score",
    "特禀质": "special_diathesis_score",
    "ADL用厕": "adl_toilet",
    "ADL吃饭": "adl_eat",
    "ADL步行": "adl_walk",
    "ADL穿衣": "adl_dress",
    "ADL洗澡": "adl_bath",
    "ADL总分": "adl_total",
    "IADL购物": "iadl_shop",
    "IADL做饭": "iadl_cook",
    "IADL理财": "iadl_finance",
    "IADL交通": "iadl_transport",
    "IADL服药": "iadl_medication",
    "IADL总分": "iadl_total",
    "活动量表总分（ADL总分+IADL总分）": "activity_total",
    "HDL-C（高密度脂蛋白）": "hdl",
    "LDL-C（低密度脂蛋白）": "ldl",
    "TG（甘油三酯）": "tg",
    "TC（总胆固醇）": "tc",
    "空腹血糖": "glucose",
    "血尿酸": "uric_acid",
    "BMI": "bmi",
    "高血脂症二分类标签": "label",
    "血脂异常分型标签（确诊病例）": "subtype",
    "年龄组": "age_group",
    "性别": "sex",
    "吸烟史": "smoke",
    "饮酒史": "drink",
}

CONSTITUTION_SCORE_COLS = [
    "balanced_score",
    "qi_deficiency_score",
    "yang_deficiency_score",
    "yin_deficiency_score",
    "phlegm_score",
    "damp_heat_score",
    "blood_stasis_score",
    "qi_stagnation_score",
    "special_diathesis_score",
]

ACTIVITY_LAB_COLS = [
    "adl_toilet",
    "adl_eat",
    "adl_walk",
    "adl_dress",
    "adl_bath",
    "adl_total",
    "iadl_shop",
    "iadl_cook",
    "iadl_finance",
    "iadl_transport",
    "iadl_medication",
    "iadl_total",
    "activity_total",
    "hdl",
    "ldl",
    "tg",
    "tc",
    "glucose",
    "uric_acid",
    "bmi",
]

ACTIVITY_ITEM_COLS = [
    "adl_toilet",
    "adl_eat",
    "adl_walk",
    "adl_dress",
    "adl_bath",
    "iadl_shop",
    "iadl_cook",
    "iadl_finance",
    "iadl_transport",
    "iadl_medication",
]
PROBLEM1_MAIN_COLS = [
    "adl_total",
    "iadl_total",
    "hdl",
    "ldl",
    "tg",
    "tc",
    "glucose",
    "uric_acid",
    "bmi",
]
PROBLEM1_ITEM_ALT_COLS = ACTIVITY_ITEM_COLS + [
    "hdl",
    "ldl",
    "tg",
    "tc",
    "glucose",
    "uric_acid",
    "bmi",
]

BASE_INFO_COLS = ["age_group", "sex", "smoke", "drink"]
DIRECT_LIPID_COLS = ["hdl", "ldl", "tg", "tc"]
MODELING_WINSOR_COLS = ["hdl", "ldl", "tg", "tc", "glucose", "uric_acid", "bmi"]
ACTIVITY_LIMIT_THRESHOLD = 8.0
WINSOR_LOWER_Q = 0.01
WINSOR_UPPER_Q = 0.99
GLUCOSE_HIGH_THRESHOLD = 6.1
BMI_HIGH_THRESHOLD = 23.9
PHLEGM_HIGH_THRESHOLD = 60.0
ACTIVITY_LOW_THRESHOLD = 60.0
URIC_ACID_HIGH_THRESHOLD = {0: 357.0, 1: 428.0}
DIAGNOSTIC_FEATURE_COLS = CONSTITUTION_SCORE_COLS + [
    "constitution_top_score",
    "constitution_margin",
    "adl_total",
    "iadl_total",
    "activity_limited_count",
    "ldl_w",
    "tg_log1p",
    "glucose_w",
    "uric_acid_log1p",
    "bmi_w",
    "non_hdl",
] + BASE_INFO_COLS
EARLY_WARNING_FEATURE_COLS = CONSTITUTION_SCORE_COLS + [
    "constitution_top_score",
    "constitution_margin",
    "adl_total",
    "iadl_total",
    "activity_limited_count",
    "glucose_w",
    "uric_acid_log1p",
    "bmi_w",
] + BASE_INFO_COLS

TCM_MONTHLY_COST = {1: 30, 2: 80, 3: 130}
TRAIN_UNIT_COST = {1: 3, 2: 5, 3: 8}
COMBO_FACTOR_COLS = [
    "lipid_abnormal",
    "uric_high",
    "glucose_high",
    "bmi_high",
    "smoke_yes",
    "drink_yes",
    "phlegm_high",
    "activity_low",
]
COMBO_FACTOR_LABELS = {
    "lipid_abnormal": "lipid_abnormal",
    "uric_high": "uric_high",
    "glucose_high": "glucose_high",
    "bmi_high": "bmi_high",
    "smoke_yes": "smoke_yes",
    "drink_yes": "drink_yes",
    "phlegm_high": "phlegm_high",
    "activity_low": "activity_low",
}


### 学习提示

当前脚本的核心设计之一，就体现在这里。

这里更强调“列名分组”，而不是散落在各处的整数列号。

建议你重点看：

1. `RAW_TO_ENGLISH` 是如何把中文列名统一成英文列名的；
2. `CONSTITUTION_SCORE_COLS / ACTIVITY_LAB_COLS` 这些列组为什么后面会反复复用。


## 3. 干预方案的数据结构

这里使用 `PlanResult` 数据类来统一保存问题 3 的优化结果。


In [ ]:
@dataclass
class PlanResult:
    sample_id: int
    age_group: int
    activity_total: float
    baseline_score: float
    target_score: float
    final_score: float
    total_cost: float
    meets_target: bool
    monthly_records: list["MonthlyRecord"]


### 函数/类签名

`class PlanResult`


### 学习提示

这段是一个类定义。

建议你先回答：

1. 它的输入是什么？
2. 它的输出是什么？
3. 它在整个脚本流程里属于哪一步？

函数签名：

`class PlanResult`


## 4. 月度动作的数据结构

这个数据类把“某个月采取什么动作”拆成了清晰字段：

- 强度；
- 每周频次；
- 对应的月度降幅；
- 对应的训练成本。


In [ ]:
@dataclass(frozen=True)
class MonthlyAction:
    intensity: int
    frequency: int
    drop_pct: int
    train_cost: int


### 函数/类签名

`class MonthlyAction`


### 学习提示

这段是一个类定义。

建议你先回答：

1. 它的输入是什么？
2. 它的输出是什么？
3. 它在整个脚本流程里属于哪一步？

函数签名：

`class MonthlyAction`


## 5. 月度执行记录的数据结构

这个数据类用来保存动态规划恢复出来的逐月路径明细。


In [ ]:
@dataclass(frozen=True)
class MonthlyRecord:
    month: int
    intensity: int
    frequency: int
    tcm_level: int
    start_score: float
    end_score: float
    tcm_cost: float
    train_cost: float


### 函数/类签名

`class MonthlyRecord`


### 学习提示

这段是一个类定义。

建议你先回答：

1. 它的输入是什么？
2. 它的输出是什么？
3. 它在整个脚本流程里属于哪一步？

函数签名：

`class MonthlyRecord`


## 类：AuditResult


In [ ]:
@dataclass
class AuditResult:
    total_rows: int
    total_cols: int
    missing_total: int
    duplicate_rows: int
    duplicate_sample_ids: int
    adl_total_match: int
    iadl_total_match: int
    activity_total_match: int
    subtype_consistent: int
    raw_constitution_tag_match: int


### 函数/类签名

`class AuditResult`


### 学习提示

这段是一个类定义。

建议你先回答：

1. 它的输入是什么？
2. 它的输出是什么？
3. 它在整个脚本流程里属于哪一步？

函数签名：

`class AuditResult`


## 类：ModelPreprocessor


In [ ]:
@dataclass
class ModelPreprocessor:
    clip_bounds: dict[str, tuple[float, float]]
    lower_q: float = WINSOR_LOWER_Q
    upper_q: float = WINSOR_UPPER_Q


### 函数/类签名

`class ModelPreprocessor`


### 学习提示

这段是一个类定义。

建议你先回答：

1. 它的输入是什么？
2. 它的输出是什么？
3. 它在整个脚本流程里属于哪一步？

函数签名：

`class ModelPreprocessor`


## 6. 自动寻找样例数据

这个函数会在当前项目目录里寻找 C 题的 `.xlsx` 样例文件。


In [ ]:
def find_default_xlsx(root: Path) -> Path:
    for path in root.rglob("*.xlsx"):
        if path.name.startswith("~$"):
            continue
        if path.parent.name.startswith("C"):
            return path
    raise FileNotFoundError("Could not find the C-problem sample workbook.")


### 函数签名

`def find_default_xlsx(root: Path)`


### 学习提示

这段是一个函数定义。

建议你先回答：

1. 它的输入是什么？
2. 它的输出是什么？
3. 它在整个脚本流程里属于哪一步？

函数签名：

`def find_default_xlsx(root: Path)`


## 7. 读取 Excel 为 DataFrame

这里直接使用 `pd.read_excel()`：

1. 读取 Excel；
2. 校验字段是否齐全；
3. 把中文列名改成英文列名；
4. 统一转成数值列。

从这里开始，后面所有建模操作都建立在 `DataFrame` 之上。


In [ ]:
def load_dataframe(path: Path) -> "pd.DataFrame":
    df = pd.read_excel(path)

    missing = [col for col in RAW_TO_ENGLISH if col not in df.columns]
    if missing:
        raise ValueError(f"Workbook is missing expected columns: {missing}")

    df = df.rename(columns=RAW_TO_ENGLISH).copy()
    keep_cols = list(RAW_TO_ENGLISH.values())
    df = df[keep_cols]
    for col in keep_cols:
        df[col] = pd.to_numeric(df[col], errors="raise")
    raw_constitution_tag = df["constitution_tag"].to_numpy().astype(int)
    corrected_constitution_tag = dominant_constitution_codes(df).astype(int)
    df.attrs["raw_constitution_tag_match"] = int(
        (raw_constitution_tag == corrected_constitution_tag).sum()
    )
    df["constitution_tag"] = corrected_constitution_tag
    return df


### 函数签名

`def load_dataframe(path: Path)`


### 学习提示

这一段是整份脚本最值得先跑通的地方。

重点关注：

1. 为什么这里直接用 `pd.read_excel()` 就够了？
2. `RAW_TO_ENGLISH` 为什么很重要？
3. 为什么这里要把所有保留列统一转成数值？

自测问题：

1. 如果 Excel 列名和预期不一致，这段代码会在哪里报错？
2. 如果某一列里混入了非数值字符串，会在哪里报错？


### 试运行：读取 DataFrame


In [ ]:
workbook = find_default_xlsx(Path.cwd())
df = load_dataframe(workbook)
print(f"Workbook: {workbook}")
print(df.shape)
print(df.head(3))
print(list(df.columns[:10]))


### 运行后你应该观察什么

重点检查：

1. DataFrame 行列数是否正确；
2. 列名是不是已经从中文变成英文；
3. 前几行数据看起来是不是正常数值。


## 函数：`dominant_constitution_codes`


In [ ]:
def dominant_constitution_codes(df: "pd.DataFrame") -> np.ndarray:
    return df[CONSTITUTION_SCORE_COLS].to_numpy().argmax(axis=1) + 1


### 函数签名

`def dominant_constitution_codes(df: 'pd.DataFrame')`


### 学习提示

这段是一个函数定义。

建议你先回答：

1. 它的输入是什么？
2. 它的输出是什么？
3. 它在整个脚本流程里属于哪一步？

函数签名：

`def dominant_constitution_codes(df: 'pd.DataFrame')`


## 函数：`audit_dataframe`


In [ ]:
def audit_dataframe(df: "pd.DataFrame") -> AuditResult:
    adl_total_match = int(
        (
            df["adl_toilet"]
            + df["adl_eat"]
            + df["adl_walk"]
            + df["adl_dress"]
            + df["adl_bath"]
            == df["adl_total"]
        ).sum()
    )
    iadl_total_match = int(
        (
            df["iadl_shop"]
            + df["iadl_cook"]
            + df["iadl_finance"]
            + df["iadl_transport"]
            + df["iadl_medication"]
            == df["iadl_total"]
        ).sum()
    )
    activity_total_match = int(((df["adl_total"] + df["iadl_total"]) == df["activity_total"]).sum())
    subtype_consistent = int((((df["label"] == 0) & (df["subtype"] == 0)) | ((df["label"] == 1) & (df["subtype"] > 0))).sum())
    raw_constitution_tag_match = int(
        df.attrs.get(
            "raw_constitution_tag_match",
            (dominant_constitution_codes(df) == df["constitution_tag"].to_numpy()).sum(),
        )
    )
    return AuditResult(
        total_rows=int(len(df)),
        total_cols=int(df.shape[1]),
        missing_total=int(df.isna().sum().sum()),
        duplicate_rows=int(df.duplicated().sum()),
        duplicate_sample_ids=int(df["sample_id"].duplicated().sum()),
        adl_total_match=adl_total_match,
        iadl_total_match=iadl_total_match,
        activity_total_match=activity_total_match,
        subtype_consistent=subtype_consistent,
        raw_constitution_tag_match=raw_constitution_tag_match,
    )


### 函数签名

`def audit_dataframe(df: 'pd.DataFrame')`


### 学习提示

这段是一个函数定义。

建议你先回答：

1. 它的输入是什么？
2. 它的输出是什么？
3. 它在整个脚本流程里属于哪一步？

函数签名：

`def audit_dataframe(df: 'pd.DataFrame')`


## 函数：`fit_model_preprocessor`


In [ ]:
def fit_model_preprocessor(train_df: "pd.DataFrame") -> ModelPreprocessor:
    clip_bounds = {}
    for col in MODELING_WINSOR_COLS:
        lower = float(train_df[col].quantile(WINSOR_LOWER_Q))
        upper = float(train_df[col].quantile(WINSOR_UPPER_Q))
        clip_bounds[col] = (lower, upper)
    return ModelPreprocessor(clip_bounds=clip_bounds)


### 函数签名

`def fit_model_preprocessor(train_df: 'pd.DataFrame')`


### 学习提示

这段是一个函数定义。

建议你先回答：

1. 它的输入是什么？
2. 它的输出是什么？
3. 它在整个脚本流程里属于哪一步？

函数签名：

`def fit_model_preprocessor(train_df: 'pd.DataFrame')`


## 函数：`prepare_modeling_dataframe`


In [ ]:
def prepare_modeling_dataframe(
    df: "pd.DataFrame", preprocessor: ModelPreprocessor
) -> "pd.DataFrame":
    engineered = df.copy()

    constitution_scores = engineered[CONSTITUTION_SCORE_COLS].to_numpy()
    sorted_scores = np.sort(constitution_scores, axis=1)
    engineered["constitution_top_score"] = sorted_scores[:, -1]
    engineered["constitution_margin"] = sorted_scores[:, -1] - sorted_scores[:, -2]
    engineered["activity_limited_count"] = (
        engineered[ACTIVITY_ITEM_COLS] < ACTIVITY_LIMIT_THRESHOLD
    ).sum(axis=1).astype(float)

    for col, (lower, upper) in preprocessor.clip_bounds.items():
        engineered[f"{col}_w"] = engineered[col].clip(lower=lower, upper=upper)

    engineered["tg_log1p"] = np.log1p(engineered["tg_w"])
    engineered["uric_acid_log1p"] = np.log1p(engineered["uric_acid_w"])
    engineered["non_hdl"] = engineered["tc_w"] - engineered["hdl_w"]
    return engineered


### 函数签名

`def prepare_modeling_dataframe(df: 'pd.DataFrame', preprocessor: ModelPreprocessor)`


### 学习提示

这段是一个函数定义。

建议你先回答：

1. 它的输入是什么？
2. 它的输出是什么？
3. 它在整个脚本流程里属于哪一步？

函数签名：

`def prepare_modeling_dataframe(df: 'pd.DataFrame', preprocessor: ModelPreprocessor)`


## 函数：`rank_problem1_candidates`


In [ ]:
def rank_problem1_candidates(
    df: "pd.DataFrame", candidate_cols: list[str]
) -> "pd.DataFrame":
    x_candidates = df[candidate_cols]
    y_phlegm = df["phlegm_score"]
    y_label = df["label"].astype(int)

    spearman_scores = {}
    for name in candidate_cols:
        spearman_scores[name] = float(spearmanr(x_candidates[name], y_phlegm).statistic)

    mi_reg = mutual_info_regression(x_candidates, y_phlegm, random_state=42)
    mi_clf = mutual_info_classif(x_candidates, y_label, random_state=42)

    rf_reg = RandomForestRegressor(n_estimators=400, random_state=42)
    rf_clf = RandomForestClassifier(n_estimators=400, random_state=42)
    rf_reg.fit(x_candidates, y_phlegm)
    rf_clf.fit(x_candidates, y_label)

    score_rows = []
    for i, name in enumerate(candidate_cols):
        score_rows.append(
            {
                "name": name,
                "spearman": spearman_scores[name],
                "spearman_abs": abs(spearman_scores[name]),
                "mi_reg": float(mi_reg[i]),
                "mi_clf": float(mi_clf[i]),
                "rf_reg": float(rf_reg.feature_importances_[i]),
                "rf_clf": float(rf_clf.feature_importances_[i]),
            }
        )
    score_df = pd.DataFrame(score_rows)

    points: dict[str, int] = {name: 0 for name in candidate_cols}
    for metric in ["spearman_abs", "mi_reg", "mi_clf", "rf_reg", "rf_clf"]:
        ordered = score_df.sort_values(metric, ascending=False)["name"].tolist()
        total = len(ordered)
        for pos, name in enumerate(ordered):
            points[name] += total - pos
        rank_map = {name: i + 1 for i, name in enumerate(ordered)}
        score_df[f"rank_{metric}"] = score_df["name"].map(rank_map)

    return (
        score_df.assign(consensus_score=score_df["name"].map(points))
        .sort_values("consensus_score", ascending=False)
        .reset_index(drop=True)
    )


### 函数签名

`def rank_problem1_candidates(df: 'pd.DataFrame', candidate_cols: list[str])`


### 学习提示

这段是一个函数定义。

建议你先回答：

1. 它的输入是什么？
2. 它的输出是什么？
3. 它在整个脚本流程里属于哪一步？

函数签名：

`def rank_problem1_candidates(df: 'pd.DataFrame', candidate_cols: list[str])`


## 8. 问题 1：关键指标筛选与体质贡献分析

这一段的实现方式很接近数据分析常见写法：

- 候选变量直接用列名选择；
- 结果也更多地组织成 `DataFrame`；
- 排序和汇总会更适合后续转表格。


In [ ]:
def summarize_problem_1(df: "pd.DataFrame") -> dict[str, object]:
    y_phlegm = df["phlegm_score"]
    y_label = df["label"].astype(int)
    consensus = rank_problem1_candidates(df, PROBLEM1_MAIN_COLS)
    item_sensitivity = rank_problem1_candidates(df, PROBLEM1_ITEM_ALT_COLS)

    base_rate = float(y_label.mean())
    tag_rows = []
    for tag in range(1, 10):
        subset = df[df["constitution_tag"] == tag]
        rate = float(subset["label"].mean())
        tag_rows.append(
            {
                "tag": tag,
                "name": CONSTITUTION_NAMES[tag - 1],
                "count": int(len(subset)),
                "positive_rate": rate,
                "lift": rate / base_rate if base_rate else 0.0,
            }
        )
    tag_summary = pd.DataFrame(tag_rows)

    constitution_scaler = StandardScaler()
    z_constitution = pd.DataFrame(
        constitution_scaler.fit_transform(df[CONSTITUTION_SCORE_COLS]),
        columns=CONSTITUTION_SCORE_COLS,
        index=df.index,
    )
    logit_x = pd.concat([z_constitution, df[BASE_INFO_COLS].astype(float)], axis=1)
    constitution_logit = LogisticRegression(max_iter=5000, random_state=42)
    constitution_logit.fit(logit_x, y_label)
    constitution_beta = constitution_logit.coef_[0][: len(CONSTITUTION_SCORE_COLS)]
    constitution_prob = constitution_logit.predict_proba(logit_x)[:, 1]

    ame_rows = []
    for i, name in enumerate(CONSTITUTION_NAMES):
        beta = float(constitution_beta[i])
        ame = float(np.mean(constitution_prob * (1 - constitution_prob) * beta))
        ame_rows.append({"name": name, "coef": beta, "ame": ame, "abs_ame": abs(ame)})
    constitution_ame = pd.DataFrame(ame_rows)
    abs_ame_sum = float(constitution_ame["abs_ame"].sum())
    constitution_ame["share"] = (
        constitution_ame["abs_ame"] / abs_ame_sum if abs_ame_sum else 0.0
    )
    constitution_ame = constitution_ame.sort_values("abs_ame", ascending=False).reset_index(
        drop=True
    )

    return {
        "consensus": consensus,
        "item_sensitivity": item_sensitivity,
        "tag_summary": tag_summary,
        "constitution_ame": constitution_ame,
    }


### 函数签名

`def summarize_problem_1(df: 'pd.DataFrame')`


### 学习提示

问题 1 的建模逻辑重点在“数据组织方式”。

重点关注：

1. `x_candidates = df[ACTIVITY_LAB_COLS]` 为什么足够直观；
2. 为什么结果最后要整理成 `DataFrame`；
3. 这样做对后续写论文表格有什么好处。


## 9. 诊断型模型特征矩阵

这个函数返回用于诊断型模型的特征矩阵。

这一步非常直观：直接用列名列表从 `df` 中取列。


In [ ]:
def build_feature_matrix(
    df: "pd.DataFrame", preprocessor: ModelPreprocessor
) -> tuple["pd.DataFrame", list[str]]:
    prepared = prepare_modeling_dataframe(df, preprocessor)
    return prepared[DIAGNOSTIC_FEATURE_COLS].copy(), list(DIAGNOSTIC_FEATURE_COLS)


### 函数签名

`def build_feature_matrix(df: 'pd.DataFrame', preprocessor: ModelPreprocessor)`


### 学习提示

这段是一个函数定义。

建议你先回答：

1. 它的输入是什么？
2. 它的输出是什么？
3. 它在整个脚本流程里属于哪一步？

函数签名：

`def build_feature_matrix(df: 'pd.DataFrame', preprocessor: ModelPreprocessor)`


## 10. 早预警模型特征矩阵

这一段依然体现“双层模型”思想：

- 诊断型模型保留直接血脂指标；
- 早预警模型去掉直接血脂指标。


In [ ]:
def build_early_warning_matrix(
    df: "pd.DataFrame", preprocessor: ModelPreprocessor
) -> tuple["pd.DataFrame", list[str]]:
    prepared = prepare_modeling_dataframe(df, preprocessor)
    return prepared[EARLY_WARNING_FEATURE_COLS].copy(), list(EARLY_WARNING_FEATURE_COLS)


### 函数签名

`def build_early_warning_matrix(df: 'pd.DataFrame', preprocessor: ModelPreprocessor)`


### 学习提示

这段是一个函数定义。

建议你先回答：

1. 它的输入是什么？
2. 它的输出是什么？
3. 它在整个脚本流程里属于哪一步？

函数签名：

`def build_early_warning_matrix(df: 'pd.DataFrame', preprocessor: ModelPreprocessor)`


## 11. 问题 2：训练双层风险模型

这是问题 2 的核心函数。

因为输入是 `DataFrame`，所以：

- 特征更容易按名称解释；
- 重要特征结果也更容易整理成表。


In [ ]:
def fit_risk_models(df: "pd.DataFrame") -> dict[str, object]:
    y = df["label"].astype(int)
    train_df, test_df, ya_train, ya_test = train_test_split(
        df, y, test_size=0.25, stratify=y, random_state=42
    )
    preprocessor = fit_model_preprocessor(train_df)

    xa_train, all_names = build_feature_matrix(train_df, preprocessor)
    xa_test, _ = build_feature_matrix(test_df, preprocessor)
    diagnostic_model = RandomForestClassifier(
        n_estimators=500, min_samples_leaf=5, random_state=42
    )
    diagnostic_model.fit(xa_train, ya_train)
    pa = diagnostic_model.predict_proba(xa_test)[:, 1]

    xe_train, ew_names = build_early_warning_matrix(train_df, preprocessor)
    xe_test, _ = build_early_warning_matrix(test_df, preprocessor)
    early_warning_model = RandomForestClassifier(
        n_estimators=500, min_samples_leaf=5, random_state=42
    )
    early_warning_model.fit(xe_train, ya_train)
    pe = early_warning_model.predict_proba(xe_test)[:, 1]

    rule_tree_names = [
        "phlegm_score",
        "activity_total",
        "hdl",
        "ldl",
        "tg",
        "tc",
        "glucose",
        "uric_acid",
        "bmi",
    ]
    rule_tree = DecisionTreeClassifier(max_depth=3, min_samples_leaf=40, random_state=42)
    rule_tree.fit(df[rule_tree_names], y)

    diagnostic_importance = (
        pd.DataFrame({"name": all_names, "importance": diagnostic_model.feature_importances_})
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )
    ew_importance = (
        pd.DataFrame({"name": ew_names, "importance": early_warning_model.feature_importances_})
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )
    risk_profile = annotate_risk_samples(df, early_warning_model, preprocessor)
    risk_counts = (
        risk_profile["risk_level"]
        .value_counts()
        .reindex(["low", "medium", "high"], fill_value=0)
        .rename_axis("level")
        .reset_index(name="count")
    )
    combo_summary = summarize_high_risk_combinations(risk_profile)

    return {
        "diagnostic_model": diagnostic_model,
        "early_warning_model": early_warning_model,
        "preprocessor": preprocessor,
        "diagnostic_auc": float(roc_auc_score(ya_test, pa)),
        "diagnostic_acc": float(accuracy_score(ya_test, pa >= 0.5)),
        "ew_auc": float(roc_auc_score(ya_test, pe)),
        "ew_acc": float(accuracy_score(ya_test, pe >= 0.5)),
        "diagnostic_importance": diagnostic_importance,
        "ew_importance": ew_importance,
        "rule_tree_text": export_text(rule_tree, feature_names=rule_tree_names),
        "risk_profile": risk_profile,
        "risk_counts": risk_counts,
        "high_risk_base_rate": combo_summary["high_risk_base_rate"],
        "combo_pairs": combo_summary["pair_summary"],
        "combo_triples": combo_summary["triple_summary"],
        "combo_all": combo_summary["all_summary"],
        "rule_driven_pair": combo_summary["rule_driven_pair"],
    }


### 函数签名

`def fit_risk_models(df: 'pd.DataFrame')`


### 学习提示

这是问题 2 最关键的函数。

建议你重点看：

1. 诊断型模型和早预警模型的输入列有什么差别；
2. 为什么重要特征这里会被整理成带列名的表；
3. 这种写法对后续出表和解释结论有什么帮助。


## 12. 统计血脂异常项数

这个函数会根据题目给定的临床阈值，计算单个患者有多少项血脂异常。


In [ ]:
def lipid_abnormal_count(row: "pd.Series") -> int:
    return int(
        (row["tc"] > 6.2)
        + (row["tg"] > 1.7)
        + (row["ldl"] > 3.1)
        + (row["hdl"] < 1.04)
    )


### 函数签名

`def lipid_abnormal_count(row: 'pd.Series')`


### 学习提示

这段是一个函数定义。

建议你先回答：

1. 它的输入是什么？
2. 它的输出是什么？
3. 它在整个脚本流程里属于哪一步？

函数签名：

`def lipid_abnormal_count(row: 'pd.Series')`


## 13. 构造单个样本的早预警输入

这里把一个 `Series` 重新包装成单行 `DataFrame`，这样就能直接喂给 sklearn 模型。


In [ ]:
def build_early_warning_row(
    row: "pd.Series", preprocessor: ModelPreprocessor
) -> "pd.DataFrame":
    row_df = pd.DataFrame([row.to_dict()])
    prepared = prepare_modeling_dataframe(row_df, preprocessor)
    return prepared[EARLY_WARNING_FEATURE_COLS]


### 函数签名

`def build_early_warning_row(row: 'pd.Series', preprocessor: ModelPreprocessor)`


### 学习提示

这段是一个函数定义。

建议你先回答：

1. 它的输入是什么？
2. 它的输出是什么？
3. 它在整个脚本流程里属于哪一步？

函数签名：

`def build_early_warning_row(row: 'pd.Series', preprocessor: ModelPreprocessor)`


## 14. 低中高风险分层

这里依然采用“规则 + 模型”的融合思路：

- 先看血脂异常项数；
- 再看痰湿积分和活动能力；
- 最后结合模型概率。


In [ ]:
def assign_risk_level(
    row: "pd.Series",
    early_warning_model: RandomForestClassifier,
    preprocessor: ModelPreprocessor,
) -> tuple[str, float]:
    prob = float(
        early_warning_model.predict_proba(build_early_warning_row(row, preprocessor))[0, 1]
    )
    phlegm = float(row["phlegm_score"])
    activity = float(row["activity_total"])
    abnormal = lipid_abnormal_count(row)

    if (abnormal >= 1 and phlegm >= 60) or (
        abnormal == 0 and phlegm >= 80 and activity < 40
    ) or prob >= 0.80:
        return "high", prob
    if abnormal >= 1 or phlegm >= 60 or (prob >= 0.45 and activity < 60):
        return "medium", prob
    return "low", prob


### 函数签名

`def assign_risk_level(row: 'pd.Series', early_warning_model: RandomForestClassifier, preprocessor: ModelPreprocessor)`


### 学习提示

这里体现的是“模型不是唯一依据”。

最终风险分层同时依赖：

- 血脂异常项数；
- 痰湿积分；
- 活动能力；
- 模型概率。

你可以边看代码边回答：为什么高风险不只看概率？


## 15. 汇总风险等级分布

这个函数会遍历整张表，为每个样本分层，然后统计低、中、高风险的样本量。


In [ ]:
def summarize_risk_levels(
    df: "pd.DataFrame",
    early_warning_model: RandomForestClassifier,
    preprocessor: ModelPreprocessor,
) -> "pd.DataFrame":
    annotated = annotate_risk_samples(df, early_warning_model, preprocessor)
    return (
        annotated["risk_level"]
        .value_counts()
        .reindex(["low", "medium", "high"], fill_value=0)
        .rename_axis("level")
        .reset_index(name="count")
    )


### 函数签名

`def summarize_risk_levels(df: 'pd.DataFrame', early_warning_model: RandomForestClassifier, preprocessor: ModelPreprocessor)`


### 学习提示

这段是一个函数定义。

建议你先回答：

1. 它的输入是什么？
2. 它的输出是什么？
3. 它在整个脚本流程里属于哪一步？

函数签名：

`def summarize_risk_levels(df: 'pd.DataFrame', early_warning_model: RandomForestClassifier, preprocessor: ModelPreprocessor)`


## 函数：`uric_acid_upper_limit`


In [ ]:
def uric_acid_upper_limit(sex: int) -> float:
    return URIC_ACID_HIGH_THRESHOLD.get(int(sex), 428.0)


### 函数签名

`def uric_acid_upper_limit(sex: int)`


### 学习提示

这段是一个函数定义。

建议你先回答：

1. 它的输入是什么？
2. 它的输出是什么？
3. 它在整个脚本流程里属于哪一步？

函数签名：

`def uric_acid_upper_limit(sex: int)`


## 函数：`annotate_risk_samples`


In [ ]:
def annotate_risk_samples(
    df: "pd.DataFrame",
    early_warning_model: RandomForestClassifier,
    preprocessor: ModelPreprocessor,
) -> "pd.DataFrame":
    annotated = df.copy()
    levels: list[str] = []
    probs: list[float] = []
    for _, row in annotated.iterrows():
        level, prob = assign_risk_level(row, early_warning_model, preprocessor)
        levels.append(level)
        probs.append(prob)

    annotated["risk_level"] = levels
    annotated["ew_prob"] = probs
    annotated["high_risk"] = annotated["risk_level"].eq("high")
    annotated["lipid_abnormal"] = annotated.apply(lipid_abnormal_count, axis=1) >= 1
    annotated["uric_high"] = annotated.apply(
        lambda row: float(row["uric_acid"]) > uric_acid_upper_limit(int(row["sex"])),
        axis=1,
    )
    annotated["glucose_high"] = annotated["glucose"] > GLUCOSE_HIGH_THRESHOLD
    annotated["bmi_high"] = annotated["bmi"] > BMI_HIGH_THRESHOLD
    annotated["smoke_yes"] = annotated["smoke"] == 1
    annotated["drink_yes"] = annotated["drink"] == 1
    annotated["phlegm_high"] = annotated["phlegm_score"] >= PHLEGM_HIGH_THRESHOLD
    annotated["activity_low"] = annotated["activity_total"] < ACTIVITY_LOW_THRESHOLD
    return annotated


### 函数签名

`def annotate_risk_samples(df: 'pd.DataFrame', early_warning_model: RandomForestClassifier, preprocessor: ModelPreprocessor)`


### 学习提示

这段是一个函数定义。

建议你先回答：

1. 它的输入是什么？
2. 它的输出是什么？
3. 它在整个脚本流程里属于哪一步？

函数签名：

`def annotate_risk_samples(df: 'pd.DataFrame', early_warning_model: RandomForestClassifier, preprocessor: ModelPreprocessor)`


## 函数：`format_combo_label`


In [ ]:
def format_combo_label(combo: tuple[str, ...]) -> str:
    return " + ".join(COMBO_FACTOR_LABELS[name] for name in combo)


### 函数签名

`def format_combo_label(combo: tuple[str, ...])`


### 学习提示

这段是一个函数定义。

建议你先回答：

1. 它的输入是什么？
2. 它的输出是什么？
3. 它在整个脚本流程里属于哪一步？

函数签名：

`def format_combo_label(combo: tuple[str, ...])`


## 函数：`summarize_high_risk_combinations`


In [ ]:
def summarize_high_risk_combinations(
    risk_df: "pd.DataFrame",
    min_high_count: int = 80,
    min_lift: float = 1.05,
    top_n: int = 5,
) -> dict[str, object]:
    high_risk_base_rate = float(risk_df["high_risk"].mean())
    high_risk_total = int(risk_df["high_risk"].sum())
    rows: list[dict[str, object]] = []

    for size in (2, 3):
        for combo in combinations(COMBO_FACTOR_COLS, size):
            mask = risk_df[list(combo)].all(axis=1)
            count_all = int(mask.sum())
            if count_all == 0:
                continue

            count_high = int((mask & risk_df["high_risk"]).sum())
            if count_high == 0:
                continue

            confidence = count_high / count_all
            lift = confidence / high_risk_base_rate if high_risk_base_rate else np.nan
            support_high = count_high / high_risk_total if high_risk_total else np.nan
            rows.append(
                {
                    "size": size,
                    "pattern_key": "|".join(combo),
                    "pattern_label": format_combo_label(combo),
                    "count_all": count_all,
                    "count_high": count_high,
                    "confidence": confidence,
                    "lift": lift,
                    "support_high": support_high,
                }
            )

    all_summary = (
        pd.DataFrame(rows)
        .sort_values(["size", "count_high", "lift", "confidence"], ascending=[True, False, False, False])
        .reset_index(drop=True)
    )
    filtered = all_summary[
        (all_summary["count_high"] >= min_high_count) & (all_summary["lift"] > min_lift)
    ].copy()
    pair_summary = (
        filtered[filtered["size"] == 2]
        .sort_values(["count_high", "lift", "confidence"], ascending=[False, False, False])
        .head(top_n)
        .reset_index(drop=True)
    )
    triple_summary = (
        filtered[filtered["size"] == 3]
        .sort_values(["count_high", "lift", "confidence"], ascending=[False, False, False])
        .head(top_n)
        .reset_index(drop=True)
    )
    rule_driven_pair = all_summary[all_summary["pattern_key"] == "lipid_abnormal|phlegm_high"]
    if rule_driven_pair.empty:
        rule_driven_pair_dict: dict[str, object] | None = None
    else:
        rule_driven_pair_dict = rule_driven_pair.iloc[0].to_dict()

    return {
        "high_risk_base_rate": high_risk_base_rate,
        "all_summary": all_summary,
        "pair_summary": pair_summary,
        "triple_summary": triple_summary,
        "rule_driven_pair": rule_driven_pair_dict,
    }


### 函数签名

`def summarize_high_risk_combinations(risk_df: 'pd.DataFrame', min_high_count: int=80, min_lift: float=1.05, top_n: int=5)`


### 学习提示

这段是一个函数定义。

建议你先回答：

1. 它的输入是什么？
2. 它的输出是什么？
3. 它在整个脚本流程里属于哪一步？

函数签名：

`def summarize_high_risk_combinations(risk_df: 'pd.DataFrame', min_high_count: int=80, min_lift: float=1.05, top_n: int=5)`


## 16. 中医调理等级映射

根据痰湿积分，把患者映射到 1 / 2 / 3 级调理方案。


In [ ]:
def tcm_level(score: float) -> int:
    if score <= 58:
        return 1
    if score <= 61:
        return 2
    return 3


### 函数签名

`def tcm_level(score: float)`


### 学习提示

这段是一个函数定义。

建议你先回答：

1. 它的输入是什么？
2. 它的输出是什么？
3. 它在整个脚本流程里属于哪一步？

函数签名：

`def tcm_level(score: float)`


## 17. 可选活动强度约束

这一段把题目里的年龄约束和活动能力约束写成代码。


In [ ]:
def allowed_intensities(age_group: int, activity_total: float) -> list[int]:
    if age_group <= 2:
        age_allowed = {1, 2, 3}
    elif age_group <= 4:
        age_allowed = {1, 2}
    else:
        age_allowed = {1}

    if activity_total < 40:
        score_allowed = {1}
    elif activity_total < 60:
        score_allowed = {1, 2}
    else:
        score_allowed = {1, 2, 3}

    return sorted(age_allowed & score_allowed)


### 函数签名

`def allowed_intensities(age_group: int, activity_total: float)`


### 学习提示

这段是一个函数定义。

建议你先回答：

1. 它的输入是什么？
2. 它的输出是什么？
3. 它在整个脚本流程里属于哪一步？

函数签名：

`def allowed_intensities(age_group: int, activity_total: float)`


## 18. 月度降分比例

这一段把题目里的“强度提升”和“频次增加”对痰湿积分的影响翻译成公式。


In [ ]:
def activity_monthly_drop(intensity: int, frequency: int) -> float:
    if frequency < 5:
        return 0.0
    return 0.03 * (intensity - 1) + 0.01 * (frequency - 5)


### 函数签名

`def activity_monthly_drop(intensity: int, frequency: int)`


### 学习提示

这段是一个函数定义。

建议你先回答：

1. 它的输入是什么？
2. 它的输出是什么？
3. 它在整个脚本流程里属于哪一步？

函数签名：

`def activity_monthly_drop(intensity: int, frequency: int)`


## 函数：`monthly_train_cost`


In [ ]:
def monthly_train_cost(intensity: int, frequency: int) -> int:
    return int(TRAIN_UNIT_COST[intensity] * frequency * 4)


### 函数签名

`def monthly_train_cost(intensity: int, frequency: int)`


### 学习提示

这段是一个函数定义。

建议你先回答：

1. 它的输入是什么？
2. 它的输出是什么？
3. 它在整个脚本流程里属于哪一步？

函数签名：

`def monthly_train_cost(intensity: int, frequency: int)`


## 19. 生成非劣月度动作集合

这个函数会先把所有允许动作列出来，再剔除“降幅相同但成本更高”的动作。


In [ ]:
def available_monthly_actions(age_group: int, activity_total: float) -> list[MonthlyAction]:
    best_by_drop: dict[int, MonthlyAction] = {}
    for intensity in allowed_intensities(age_group, activity_total):
        for frequency in range(1, 11):
            drop_pct = int(round(activity_monthly_drop(intensity, frequency) * 100))
            candidate = MonthlyAction(
                intensity=intensity,
                frequency=frequency,
                drop_pct=drop_pct,
                train_cost=monthly_train_cost(intensity, frequency),
            )
            current = best_by_drop.get(drop_pct)
            if current is None:
                best_by_drop[drop_pct] = candidate
                continue
            if candidate.train_cost < current.train_cost:
                best_by_drop[drop_pct] = candidate
                continue
            if candidate.train_cost == current.train_cost and (
                candidate.intensity,
                candidate.frequency,
            ) < (
                current.intensity,
                current.frequency,
            ):
                best_by_drop[drop_pct] = candidate
    return [best_by_drop[key] for key in sorted(best_by_drop)]


### 函数签名

`def available_monthly_actions(age_group: int, activity_total: float)`


### 学习提示

这段是一个函数定义。

建议你先回答：

1. 它的输入是什么？
2. 它的输出是什么？
3. 它在整个脚本流程里属于哪一步？

函数签名：

`def available_monthly_actions(age_group: int, activity_total: float)`


## 20. 模拟一条月度路径

给定一串按月动作，这个函数会逐月模拟痰湿积分变化和成本变化。


In [ ]:
def simulate_action_sequence(
    baseline_score: float,
    action_sequence: list[MonthlyAction],
) -> tuple[float, float, list[MonthlyRecord]]:
    score = float(baseline_score)
    total_cost = 0.0
    history: list[MonthlyRecord] = []

    for month, action in enumerate(action_sequence, start=1):
        start_score = score
        current_tcm_level = tcm_level(start_score)
        tcm_cost = TCM_MONTHLY_COST[current_tcm_level]
        train_cost = float(action.train_cost)
        total_cost += tcm_cost + train_cost
        score = score * (1 - action.drop_pct / 100.0)
        history.append(
            MonthlyRecord(
                month=month,
                intensity=action.intensity,
                frequency=action.frequency,
                tcm_level=current_tcm_level,
                start_score=round(start_score, 3),
                end_score=round(score, 3),
                tcm_cost=float(tcm_cost),
                train_cost=train_cost,
            )
        )

    return score, total_cost, history


### 函数签名

`def simulate_action_sequence(baseline_score: float, action_sequence: list[MonthlyAction])`


### 学习提示

这段是一个函数定义。

建议你先回答：

1. 它的输入是什么？
2. 它的输出是什么？
3. 它在整个脚本流程里属于哪一步？

函数签名：

`def simulate_action_sequence(baseline_score: float, action_sequence: list[MonthlyAction])`


## 21. 问题 3：搜索最优干预方案

这一步不再固定 6 个月都用同一方案，而是把“月份 + 当前积分”作为状态，用动态规划求最小总成本。


In [ ]:
def optimize_intervention(
    row: "pd.Series",
    target_ratio: float = 0.90,
    budget: float = 2000.0,
) -> PlanResult:
    sample_id = int(row["sample_id"])
    age_group = int(row["age_group"])
    activity_total = float(row["activity_total"])
    baseline_score = float(row["phlegm_score"])
    target_score = baseline_score * target_ratio
    months = 6
    score_scale = 1000
    inf_cost = 10**9
    budget_int = int(round(budget))
    actions = available_monthly_actions(age_group, activity_total)
    start_idx = int(round(baseline_score * score_scale))
    target_idx = int(np.floor(target_score * score_scale + 1e-9))
    state_grid = np.arange(start_idx + 1, dtype=np.int32)
    tcm_cost_by_state = np.where(
        state_grid <= 58 * score_scale,
        TCM_MONTHLY_COST[1],
        np.where(state_grid <= 61 * score_scale, TCM_MONTHLY_COST[2], TCM_MONTHLY_COST[3]),
    ).astype(np.int32)
    next_state_lookup = np.empty((len(actions), start_idx + 1), dtype=np.int32)
    for action_idx, action in enumerate(actions):
        next_state_lookup[action_idx] = (
            state_grid * (100 - action.drop_pct) + 50
        ) // 100

    current_cost = np.full(start_idx + 1, inf_cost, dtype=np.int32)
    current_cost[start_idx] = 0
    active_states = np.array([start_idx], dtype=np.int32)
    parent_states: list[np.ndarray] = []
    parent_actions: list[np.ndarray] = []

    for _ in range(months):
        next_cost = np.full(start_idx + 1, inf_cost, dtype=np.int32)
        parent_state = np.full(start_idx + 1, -1, dtype=np.int32)
        parent_action = np.full(start_idx + 1, -1, dtype=np.int16)

        for idx in active_states.tolist():
            month_base_cost = int(current_cost[idx]) + int(tcm_cost_by_state[idx])
            for action_idx, action in enumerate(actions):
                nxt = int(next_state_lookup[action_idx, idx])
                candidate_cost = month_base_cost + int(action.train_cost)
                if candidate_cost < next_cost[nxt]:
                    next_cost[nxt] = candidate_cost
                    parent_state[nxt] = idx
                    parent_action[nxt] = action_idx

        parent_states.append(parent_state)
        parent_actions.append(parent_action)
        current_cost = next_cost
        active_states = np.flatnonzero(current_cost < inf_cost).astype(np.int32)

    def reconstruct_actions(final_state: int) -> list[MonthlyAction]:
        sequence: list[MonthlyAction] = []
        state = int(final_state)
        for month in range(months - 1, -1, -1):
            action_idx = int(parent_actions[month][state])
            if action_idx < 0:
                raise ValueError(f"State {final_state} is not reachable in month {month + 1}.")
            sequence.append(actions[action_idx])
            state = int(parent_states[month][state])
        sequence.reverse()
        return sequence

    def build_result(final_state: int) -> PlanResult:
        sequence = reconstruct_actions(final_state)
        final_score, total_cost, history = simulate_action_sequence(baseline_score, sequence)
        meets_target = final_score <= target_score + 1e-9 and total_cost <= budget + 1e-9
        return PlanResult(
            sample_id=sample_id,
            age_group=age_group,
            activity_total=activity_total,
            baseline_score=baseline_score,
            target_score=target_score,
            final_score=final_score,
            total_cost=total_cost,
            meets_target=meets_target,
            monthly_records=history,
        )

    final_cost = current_cost
    reachable_states = np.flatnonzero(final_cost < inf_cost).astype(np.int32)
    target_affordable = [
        int(idx)
        for idx in reachable_states
        if idx <= target_idx and int(final_cost[idx]) <= budget_int
    ]
    if target_affordable:
        ordered_states = sorted(target_affordable, key=lambda idx: (int(final_cost[idx]), idx))
    else:
        affordable_states = [
            int(idx) for idx in reachable_states if int(final_cost[idx]) <= budget_int
        ]
        if affordable_states:
            ordered_states = sorted(affordable_states, key=lambda idx: (idx, int(final_cost[idx])))
        else:
            ordered_states = sorted(reachable_states.tolist(), key=lambda idx: (int(final_cost[idx]), idx))

    for final_state in ordered_states:
        result = build_result(final_state)
        if target_affordable:
            if result.meets_target:
                return result
            continue
        return result

    raise RuntimeError("Failed to construct an intervention plan.")


### 函数签名

`def optimize_intervention(row: 'pd.Series', target_ratio: float=0.9, budget: float=2000.0)`


### 学习提示

问题 3 的优化逻辑重点已经变成“状态设计 + 动态规划”。

建议你关注：

1. 这里输入为什么选择 `pd.Series`；
2. 状态为什么只保留“当前月份 + 当前积分”就够了；
3. 为什么要先剔除同降幅高成本的动作。


## 24. 问题 1 输出函数

把问题 1 的核心结果打印得更清晰。


In [ ]:
def print_problem_1(summary: dict[str, object]) -> None:
    print("=== Problem 1: main-analysis indicators ===")
    for _, row in summary["consensus"].iterrows():
        print(
            f"{row['name']:<22} "
            f"spearman={row['spearman']:+.3f} "
            f"mi_phlegm={row['mi_reg']:.3f} "
            f"mi_label={row['mi_clf']:.3f}"
        )

    print("\nSupplementary mixed item-level analysis")
    for _, row in summary["item_sensitivity"].head(10).iterrows():
        print(
            f"{row['name']:<22} "
            f"spearman={row['spearman']:+.3f} "
            f"mi_phlegm={row['mi_reg']:.3f} "
            f"mi_label={row['mi_clf']:.3f}"
        )

    print("\nDominant constitution positive rates")
    for _, row in summary["tag_summary"].iterrows():
        print(
            f"{int(row['tag'])}. {row['name']:<18} "
            f"n={int(row['count']):<3d} "
            f"rate={row['positive_rate']:.3f} "
            f"lift={row['lift']:.3f}"
        )

    print("\nModel-based constitution contribution (standardized logit + AME)")
    for _, row in summary["constitution_ame"].iterrows():
        print(
            f"{row['name']:<18} "
            f"coef={row['coef']:+.3f} "
            f"ame={row['ame']:+.4f} "
            f"share={row['share']:.3f}"
        )


### 函数签名

`def print_problem_1(summary: dict[str, object])`


### 学习提示

这段是一个函数定义。

建议你先回答：

1. 它的输入是什么？
2. 它的输出是什么？
3. 它在整个脚本流程里属于哪一步？

函数签名：

`def print_problem_1(summary: dict[str, object])`


### 试运行：问题 1


In [ ]:
problem_1 = summarize_problem_1(df)
print_problem_1(problem_1)
print(problem_1["consensus"].head())


### 运行后你应该观察什么

重点看：

1. 共识排序前几名是谁；
2. `consensus` 已经是 DataFrame，这对后面导出表格非常方便。


## 函数：`print_preprocessing_audit`


In [ ]:
def print_preprocessing_audit(audit: AuditResult) -> None:
    print("=== Data audit ===")
    print(
        f"missing_total={audit.missing_total}, duplicate_rows={audit.duplicate_rows}, "
        f"duplicate_sample_ids={audit.duplicate_sample_ids}"
    )
    print(
        f"adl_total_consistency={audit.adl_total_match}/{audit.total_rows}, "
        f"iadl_total_consistency={audit.iadl_total_match}/{audit.total_rows}, "
        f"activity_total_consistency={audit.activity_total_match}/{audit.total_rows}"
    )
    print(
        f"label_subtype_consistency={audit.subtype_consistent}/{audit.total_rows}, "
        f"raw_constitution_tag_match={audit.raw_constitution_tag_match}/{audit.total_rows}, "
        f"raw_constitution_tag_corrected={audit.total_rows - audit.raw_constitution_tag_match}"
    )


### 函数签名

`def print_preprocessing_audit(audit: AuditResult)`


### 学习提示

这段是一个函数定义。

建议你先回答：

1. 它的输入是什么？
2. 它的输出是什么？
3. 它在整个脚本流程里属于哪一步？

函数签名：

`def print_preprocessing_audit(audit: AuditResult)`


## 25. 问题 2 输出函数

把问题 2 的模型表现、重要特征和分层结果集中打印出来。


In [ ]:
def print_problem_2(summary: dict[str, object], df: "pd.DataFrame") -> None:
    print("\n=== Problem 2: risk model ===")
    print(
        f"Train-based winsorization ({summary['preprocessor'].lower_q:.0%}-{summary['preprocessor'].upper_q:.0%}) "
        f"applied to: {', '.join(MODELING_WINSOR_COLS)}"
    )
    print(
        "Derived features: tg_log1p, uric_acid_log1p, non_hdl, "
        "activity_limited_count, "
        "constitution_top_score, constitution_margin"
    )
    print(
        f"Diagnostic model   AUC={summary['diagnostic_auc']:.4f} "
        f"ACC={summary['diagnostic_acc']:.4f}"
    )
    print(
        f"Early warning model AUC={summary['ew_auc']:.4f} "
        f"ACC={summary['ew_acc']:.4f}"
    )

    print("\nTop diagnostic features")
    for _, row in summary["diagnostic_importance"].head(10).iterrows():
        print(f"{row['name']:<22} {row['importance']:.4f}")

    print("\nTop early warning features")
    for _, row in summary["ew_importance"].head(10).iterrows():
        print(f"{row['name']:<22} {row['importance']:.4f}")

    print("\nRule tree with direct lipids")
    print(summary["rule_tree_text"].strip())

    print("Assigned low/medium/high counts")
    for _, row in summary["risk_counts"].iterrows():
        print(f"{row['level']:<6} {int(row['count'])}")

    print(
        f"\nHigh-risk base rate = {summary['high_risk_base_rate']:.3f} "
        "(used for combination lift)"
    )
    print("Top pairwise high-risk combinations")
    for _, row in summary["combo_pairs"].iterrows():
        print(
            f"{row['pattern_label']:<45} "
            f"high={int(row['count_high']):<3d} "
            f"conf={row['confidence']:.3f} "
            f"lift={row['lift']:.3f}"
        )

    print("\nTop triple high-risk combinations")
    for _, row in summary["combo_triples"].iterrows():
        print(
            f"{row['pattern_label']:<45} "
            f"high={int(row['count_high']):<3d} "
            f"conf={row['confidence']:.3f} "
            f"lift={row['lift']:.3f}"
        )

    if summary["rule_driven_pair"] is not None:
        row = summary["rule_driven_pair"]
        print(
            "\nRule-driven pattern check\n"
            f"{row['pattern_label']:<45} "
            f"high={int(row['count_high']):<3d} "
            f"conf={row['confidence']:.3f} "
            f"lift={row['lift']:.3f}"
        )


### 函数签名

`def print_problem_2(summary: dict[str, object], df: 'pd.DataFrame')`


### 学习提示

这段是一个函数定义。

建议你先回答：

1. 它的输入是什么？
2. 它的输出是什么？
3. 它在整个脚本流程里属于哪一步？

函数签名：

`def print_problem_2(summary: dict[str, object], df: 'pd.DataFrame')`


### 试运行：问题 2


In [ ]:
problem_2 = fit_risk_models(df)
print_problem_2(problem_2, df)
print(problem_2["diagnostic_importance"].head())


### 运行后你应该观察什么

重点看：

1. 两套模型效果差多少；
2. 重要特征表是否已经带有清晰列名；
3. 风险等级统计是否符合预期。


## 26. 问题 3 输出函数

把指定样本的最优方案和逐月明细打印出来。


In [ ]:
def print_problem_3(
    df: "pd.DataFrame",
    sample_ids: list[int],
    target_ratio: float,
    budget: float,
) -> None:
    print("\n=== Problem 3: intervention optimization ===")
    for sample_id in sample_ids:
        matches = df[df["sample_id"] == sample_id]
        if matches.empty:
            print(f"Sample {sample_id} not found.")
            continue

        result = optimize_intervention(matches.iloc[0], target_ratio=target_ratio, budget=budget)
        segments = []
        if result.monthly_records:
            seg_start = result.monthly_records[0].month
            seg_action = (
                result.monthly_records[0].intensity,
                result.monthly_records[0].frequency,
            )
            prev_month = result.monthly_records[0].month
            for record in result.monthly_records[1:]:
                action = (record.intensity, record.frequency)
                if action != seg_action:
                    label = (
                        f"M{seg_start}"
                        if seg_start == prev_month
                        else f"M{seg_start}-M{prev_month}"
                    )
                    segments.append(
                        f"{label}: intensity={seg_action[0]}, freq={seg_action[1]}/week"
                    )
                    seg_start = record.month
                    seg_action = action
                prev_month = record.month
            label = f"M{seg_start}" if seg_start == prev_month else f"M{seg_start}-M{prev_month}"
            segments.append(f"{label}: intensity={seg_action[0]}, freq={seg_action[1]}/week")
        print(
            f"Sample {sample_id}: baseline={result.baseline_score:.1f}, "
            f"target<={result.target_score:.1f}, activity_total={result.activity_total:.1f}, "
            f"age_group={result.age_group}"
        )
        print(
            f"  dynamic plan: {'; '.join(segments)}, "
            f"final={result.final_score:.2f}, cost={result.total_cost:.0f}, "
            f"meets_target={result.meets_target}"
        )
        for record in result.monthly_records:
            print(
                f"  month {record.month}: intensity={record.intensity}, "
                f"freq={record.frequency}/week, tcm_level={record.tcm_level}, "
                f"start={record.start_score:.3f}, end={record.end_score:.3f}, "
                f"tcm_cost={record.tcm_cost:.0f}, train_cost={record.train_cost:.0f}"
            )


### 函数签名

`def print_problem_3(df: 'pd.DataFrame', sample_ids: list[int], target_ratio: float, budget: float)`


### 学习提示

这段是一个函数定义。

建议你先回答：

1. 它的输入是什么？
2. 它的输出是什么？
3. 它在整个脚本流程里属于哪一步？

函数签名：

`def print_problem_3(df: 'pd.DataFrame', sample_ids: list[int], target_ratio: float, budget: float)`


### 试运行：问题 3


In [ ]:
print_problem_3(df, [1, 2, 3], target_ratio=0.90, budget=2000.0)


### 运行后你应该观察什么

重点看：

1. DataFrame 版在读取单个患者属性时是不是更直观；
2. 三个样本的最终方案是否满足目标并控制在预算内。


## 函数：`summarize_plan_segments`


In [ ]:
def summarize_plan_segments(result: PlanResult) -> list[dict[str, object]]:
    segments: list[dict[str, object]] = []
    if not result.monthly_records:
        return segments

    seg_start = result.monthly_records[0].month
    seg_action = (
        result.monthly_records[0].intensity,
        result.monthly_records[0].frequency,
    )
    prev_month = result.monthly_records[0].month
    for record in result.monthly_records[1:]:
        action = (record.intensity, record.frequency)
        if action != seg_action:
            segments.append(
                {
                    "month_from": seg_start,
                    "month_to": prev_month,
                    "intensity": seg_action[0],
                    "frequency": seg_action[1],
                    "label": (
                        f"M{seg_start}"
                        if seg_start == prev_month
                        else f"M{seg_start}-M{prev_month}"
                    ),
                }
            )
            seg_start = record.month
            seg_action = action
        prev_month = record.month

    segments.append(
        {
            "month_from": seg_start,
            "month_to": prev_month,
            "intensity": seg_action[0],
            "frequency": seg_action[1],
            "label": f"M{seg_start}" if seg_start == prev_month else f"M{seg_start}-M{prev_month}",
        }
    )
    return segments


### 函数签名

`def summarize_plan_segments(result: PlanResult)`


### 学习提示

这段是一个函数定义。

建议你先回答：

1. 它的输入是什么？
2. 它的输出是什么？
3. 它在整个脚本流程里属于哪一步？

函数签名：

`def summarize_plan_segments(result: PlanResult)`


## 22. 批量收集问题 3 结果

这个函数会对指定样本列表逐个求解最优方案。


In [ ]:
def collect_problem_3_results(
    df: "pd.DataFrame",
    sample_ids: list[int],
    target_ratio: float,
    budget: float,
) -> list[PlanResult]:
    results: list[PlanResult] = []
    for sample_id in sample_ids:
        matches = df[df["sample_id"] == sample_id]
        if matches.empty:
            continue
        results.append(optimize_intervention(matches.iloc[0], target_ratio=target_ratio, budget=budget))
    return results


### 函数签名

`def collect_problem_3_results(df: 'pd.DataFrame', sample_ids: list[int], target_ratio: float, budget: float)`


### 学习提示

这段是一个函数定义。

建议你先回答：

1. 它的输入是什么？
2. 它的输出是什么？
3. 它在整个脚本流程里属于哪一步？

函数签名：

`def collect_problem_3_results(df: 'pd.DataFrame', sample_ids: list[int], target_ratio: float, budget: float)`


## 23. 导出三道题结果

这个函数会把问题 1 / 2 / 3 的核心结果自动写入 `赛题/outputs/`。


In [ ]:
def export_problem_outputs(
    workbook: Path,
    audit: AuditResult,
    problem_1: dict[str, object],
    problem_2: dict[str, object],
    problem_3_results: list[PlanResult],
    output_root: Path,
) -> None:
    output_root.mkdir(parents=True, exist_ok=True)
    problem1_dir = output_root / "problem1"
    problem2_dir = output_root / "problem2"
    problem3_dir = output_root / "problem3"
    for path in (problem1_dir, problem2_dir, problem3_dir):
        path.mkdir(parents=True, exist_ok=True)

    audit_payload = {
        "workbook": str(workbook),
        "total_rows": audit.total_rows,
        "total_cols": audit.total_cols,
        "missing_total": audit.missing_total,
        "duplicate_rows": audit.duplicate_rows,
        "duplicate_sample_ids": audit.duplicate_sample_ids,
        "adl_total_match": audit.adl_total_match,
        "iadl_total_match": audit.iadl_total_match,
        "activity_total_match": audit.activity_total_match,
        "subtype_consistent": audit.subtype_consistent,
        "raw_constitution_tag_match": audit.raw_constitution_tag_match,
    }
    (output_root / "audit.json").write_text(
        json.dumps(audit_payload, ensure_ascii=False, indent=2), encoding="utf-8"
    )

    problem_1["consensus"].to_csv(problem1_dir / "main_consensus.csv", index=False, encoding="utf-8-sig")
    problem_1["item_sensitivity"].to_csv(
        problem1_dir / "item_sensitivity.csv", index=False, encoding="utf-8-sig"
    )
    problem_1["tag_summary"].to_csv(problem1_dir / "tag_summary.csv", index=False, encoding="utf-8-sig")
    problem_1["constitution_ame"].to_csv(
        problem1_dir / "constitution_ame.csv", index=False, encoding="utf-8-sig"
    )

    metrics_payload = {
        "diagnostic_auc": problem_2["diagnostic_auc"],
        "diagnostic_acc": problem_2["diagnostic_acc"],
        "early_warning_auc": problem_2["ew_auc"],
        "early_warning_acc": problem_2["ew_acc"],
        "high_risk_base_rate": problem_2["high_risk_base_rate"],
        "winsor_lower_q": problem_2["preprocessor"].lower_q,
        "winsor_upper_q": problem_2["preprocessor"].upper_q,
        "winsor_columns": MODELING_WINSOR_COLS,
    }
    (problem2_dir / "metrics.json").write_text(
        json.dumps(metrics_payload, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    problem_2["diagnostic_importance"].to_csv(
        problem2_dir / "diagnostic_importance.csv", index=False, encoding="utf-8-sig"
    )
    problem_2["ew_importance"].to_csv(
        problem2_dir / "early_warning_importance.csv", index=False, encoding="utf-8-sig"
    )
    problem_2["risk_counts"].to_csv(problem2_dir / "risk_counts.csv", index=False, encoding="utf-8-sig")
    problem_2["risk_profile"].to_csv(problem2_dir / "risk_profile.csv", index=False, encoding="utf-8-sig")
    problem_2["combo_pairs"].to_csv(problem2_dir / "combo_pairs.csv", index=False, encoding="utf-8-sig")
    problem_2["combo_triples"].to_csv(
        problem2_dir / "combo_triples.csv", index=False, encoding="utf-8-sig"
    )
    problem_2["combo_all"].to_csv(problem2_dir / "combo_all.csv", index=False, encoding="utf-8-sig")
    (problem2_dir / "rule_tree.txt").write_text(problem_2["rule_tree_text"], encoding="utf-8")
    if problem_2["rule_driven_pair"] is not None:
        (problem2_dir / "rule_driven_pair.json").write_text(
            json.dumps(problem_2["rule_driven_pair"], ensure_ascii=False, indent=2),
            encoding="utf-8",
        )

    report_figure_dir = Path(__file__).resolve().parents[2] / "论文" / "Figures"
    if report_figure_dir.exists():
        figure_dir = problem2_dir / "figures"
        figure_dir.mkdir(parents=True, exist_ok=True)
        for name in [
            "q2_roc_curve.png",
            "q2_diag_importance.png",
            "q2_ew_importance.png",
            "q2_diag_shap_beeswarm.png",
            "q2_ew_shap_beeswarm.png",
            "q2_ew_shap_waterfall.png",
            "q2_combo_patterns.png",
            "q2_model_visuals.png",
        ]:
            src = report_figure_dir / name
            if src.exists():
                shutil.copy2(src, figure_dir / name)

    plan_rows: list[dict[str, object]] = []
    monthly_rows: list[dict[str, object]] = []
    for result in problem_3_results:
        segments = summarize_plan_segments(result)
        plan_rows.append(
            {
                "sample_id": result.sample_id,
                "age_group": result.age_group,
                "activity_total": result.activity_total,
                "baseline_score": result.baseline_score,
                "target_score": result.target_score,
                "final_score": round(result.final_score, 6),
                "total_cost": round(result.total_cost, 2),
                "meets_target": result.meets_target,
                "plan_segments": "; ".join(
                    f"{seg['label']}:({seg['intensity']},{seg['frequency']})" for seg in segments
                ),
            }
        )
        for seg in segments:
            monthly_rows.append(
                {
                    "sample_id": result.sample_id,
                    "row_type": "segment",
                    "month": None,
                    "month_from": seg["month_from"],
                    "month_to": seg["month_to"],
                    "intensity": seg["intensity"],
                    "frequency": seg["frequency"],
                    "tcm_level": None,
                    "start_score": None,
                    "end_score": None,
                    "tcm_cost": None,
                    "train_cost": None,
                }
            )
        for record in result.monthly_records:
            monthly_rows.append(
                {
                    "sample_id": result.sample_id,
                    "row_type": "month",
                    "month": record.month,
                    "month_from": record.month,
                    "month_to": record.month,
                    "intensity": record.intensity,
                    "frequency": record.frequency,
                    "tcm_level": record.tcm_level,
                    "start_score": record.start_score,
                    "end_score": record.end_score,
                    "tcm_cost": record.tcm_cost,
                    "train_cost": record.train_cost,
                }
            )

    pd.DataFrame(plan_rows).to_csv(
        problem3_dir / "sample_plans_summary.csv", index=False, encoding="utf-8-sig"
    )
    pd.DataFrame(monthly_rows).to_csv(
        problem3_dir / "sample_plans_monthly.csv", index=False, encoding="utf-8-sig"
    )

    readme = """# C题输出结果

- `audit.json`: 数据核验结果
- `problem1/`: 问题一的综合排序、体质分组和 AME 结果
- `problem2/`: 问题二的模型指标、风险分层、组合模式和图像副本
- `problem3/`: 问题三的样本方案汇总与逐月明细
"""
    (output_root / "README.md").write_text(readme, encoding="utf-8")


### 函数签名

`def export_problem_outputs(workbook: Path, audit: AuditResult, problem_1: dict[str, object], problem_2: dict[str, object], problem_3_results: list[PlanResult], output_root: Path)`


### 学习提示

这一段很适合用来学习“研究脚本如何落地成可交付结果”。

重点看：

1. 为什么每一道题都单独建子目录；
2. 为什么有的结果适合保存成 `csv`，有的适合保存成 `json/txt`；
3. 图像结果是怎样从论文目录复制到 `outputs` 的。


## 27. 命令行入口

如果你在终端里直接运行 `c_problem_starter.py`，就是从这里开始执行。


In [ ]:
def main() -> None:
    parser = argparse.ArgumentParser(description="Pandas-based starter analysis for the MathorCup C problem.")
    parser.add_argument("--xlsx", type=Path, default=None, help="Path to the sample workbook.")
    parser.add_argument("--budget", type=float, default=2000.0, help="6-month intervention budget.")
    parser.add_argument(
        "--target-ratio",
        type=float,
        default=0.90,
        help="Target final phlegm score as a ratio of baseline score.",
    )
    parser.add_argument(
        "--sample-ids",
        type=int,
        nargs="*",
        default=[1, 2, 3],
        help="Sample IDs for the intervention section.",
    )
    parser.add_argument(
        "--output-dir",
        type=Path,
        default=Path(__file__).resolve().parent.parent / "outputs",
        help="Directory for exported problem outputs.",
    )
    args = parser.parse_args()

    workbook = args.xlsx or find_default_xlsx(Path.cwd())
    df = load_dataframe(workbook)
    audit = audit_dataframe(df)
    print(f"Workbook: {workbook}")
    print(f"Rows={len(df)}, Cols={df.shape[1]}")
    print_preprocessing_audit(audit)

    problem_1 = summarize_problem_1(df)
    problem_2 = fit_risk_models(df)
    problem_3_results = collect_problem_3_results(df, args.sample_ids, args.target_ratio, args.budget)

    print_problem_1(problem_1)
    print_problem_2(problem_2, df)
    print_problem_3(df, args.sample_ids, args.target_ratio, args.budget)
    export_problem_outputs(workbook, audit, problem_1, problem_2, problem_3_results, args.output_dir)
    print(f"\nExported outputs to: {args.output_dir}")


### 函数签名

`def main()`


### 学习提示

你可以把 `main()` 当成整份脚本的“流程图”来看。

如果你只想研究某一问，其实不一定要跑它；你可以直接在 notebook 中单独调用对应函数。


## 28. 入口判断

这一段是标准 Python 脚本入口判断，在 notebook 里一般不会直接依赖，但保留有助于理解脚本结构。


In [ ]:
if __name__ == "__main__":
    main()


### 学习提示

脚本模式下会从这里触发 `main()`；
而在 notebook 中，你通常会更喜欢逐段调用前面的函数。


## 结束说明

这份 notebook 适合你按“读数据 -> 做筛选 -> 做预警 -> 做优化”的顺序反复回看。

如果你之后继续修改 `c_problem_starter.py`，重新运行本目录下的 `build_step_by_step_notebook.py` 即可自动重建 notebook。
